# Notebook 18
## DNA Nucleotide Transformer Downstream Classifiers

Trains and evaluates classical ML classifiers on top of the frozen Nucleotide
Transformer embeddings produced by notebook 17.

**Task:** promoter vs non-promoter (binary classification)
**Embedding dim:** 1280 (500M human-ref model)

**Models:** Logistic Regression, Linear SVM, Random Forest, XGBoost

**Split:** identical `train_test_split(random_state=SEED, test_size=0.2, stratify=y)`
used across all prior DNA notebooks.

### Outputs
- `reports/dna_nt_test_results.csv`
- `reports/dna_nt_models_summary.json`
- `reports/figures/dna_nt/cm_<model>.png`
- `reports/figures/dna_nt/roc_<model>.png`
- `models/dna/nt/<model>.pkl`
- `reports/dna_all_paradigms_comparison.csv` (updated)

## 1) Imports

In [1]:
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve,
)

warnings.filterwarnings('ignore')
np.set_printoptions(suppress=True)
print('imports OK')


imports OK


## 2) Paths, config, seed

In [2]:
ROOT      = Path.cwd().parents[0]
PROCESSED = ROOT / 'data' / 'processed'
REPORTS   = ROOT / 'reports'
FIGURES   = REPORTS / 'figures' / 'dna_nt'
MODELS    = ROOT / 'models' / 'dna' / 'nt'
CONFIGS   = ROOT / 'configs'

for p in [REPORTS, FIGURES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)

with open(CONFIGS / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

SEED  = int(cfg['project']['random_seed'])
L     = int(cfg['dna']['seq_length_bp'])
N_POS = int(cfg['dna']['n_pos'])
N_NEG = int(cfg['dna']['n_neg'])

random.seed(SEED)
np.random.seed(SEED)
print(f'SEED={SEED}  L={L}  N_POS={N_POS}  N_NEG={N_NEG}')


SEED=42  L=200  N_POS=2000  N_NEG=2000


## 3) Load NT embeddings

Load the three parallel arrays saved by notebook 17 in original row order.

In [3]:
sfx = f'len{L}_pos{N_POS}_neg{N_NEG}'

emb_path    = PROCESSED / f'dna_nt_embeddings_{sfx}.npy'
labels_path = PROCESSED / f'dna_nt_labels_{sfx}.npy'
ids_path    = PROCESSED / f'dna_nt_ids_{sfx}.npy'

for p in [emb_path, labels_path, ids_path]:
    assert p.exists(), f'Missing: {p}  -- run notebook 17 first'

X   = np.load(emb_path)
y   = np.load(labels_path)
ids = np.load(ids_path, allow_pickle=True)

print(f'X shape:  {X.shape}   dtype: {X.dtype}')
print(f'y shape:  {y.shape}   classes: {dict(zip(*np.unique(y, return_counts=True)))}')
assert not np.isnan(X).any() and not np.isinf(X).any()
assert X.shape[0] == y.shape[0] == ids.shape[0]
print('Integrity checks passed.')


X shape:  (4000, 1280)   dtype: float32
y shape:  (4000,)   classes: {np.int64(0): np.int64(2000), np.int64(1): np.int64(2000)}
Integrity checks passed.


## 4) Train / test split

Same parameters as every other DNA notebook: `test_size=0.2, random_state=SEED,
stratify=y`. This yields the same 800-sample holdout set.

In [4]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, ids,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print(f'Train: {X_train.shape}  balance: {y_train.mean():.3f}')
print(f'Test:  {X_test.shape}   balance: {y_test.mean():.3f}')


Train: (3200, 1280)  balance: 0.500
Test:  (800, 1280)   balance: 0.500


## 5) Define classifiers

Identical definitions to notebooks 06 and 16 for direct comparability.
NT embeddings are not unit-normalised (std ~0.83 from the summary), so
scaling is important for LR and SVM.

In [5]:
models = {}

models['logreg'] = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=5000, solver='lbfgs',
                               random_state=SEED)),
])

models['linear_svm_calibrated'] = CalibratedClassifierCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('svm', LinearSVC(random_state=SEED, max_iter=5000)),
    ]),
    method='sigmoid', cv=3,
)

models['random_forest'] = RandomForestClassifier(
    n_estimators=400, random_state=SEED, n_jobs=-1,
)

try:
    from xgboost import XGBClassifier
    models['xgboost'] = XGBClassifier(
        n_estimators=600, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, eval_metric='logloss',
        tree_method='hist',
    )
except ImportError:
    print('XGBoost not available, skipping.')

print('Models defined:', list(models.keys()))


Models defined: ['logreg', 'linear_svm_calibrated', 'random_forest', 'xgboost']


## 6) 5-fold cross-validation on training set

In [6]:
CV_FOLDS = 5
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)

cv_results = []
for name, clf in models.items():
    print(f'CV: {name} ...', end=' ', flush=True)
    scores = cross_validate(
        clf, X_train, y_train,
        cv=skf,
        scoring=['accuracy', 'f1', 'roc_auc'],
        n_jobs=1,
    )
    row = {
        'model':            name,
        'cv_accuracy_mean': scores['test_accuracy'].mean(),
        'cv_accuracy_std':  scores['test_accuracy'].std(),
        'cv_f1_mean':       scores['test_f1'].mean(),
        'cv_f1_std':        scores['test_f1'].std(),
        'cv_roc_auc_mean':  scores['test_roc_auc'].mean(),
        'cv_roc_auc_std':   scores['test_roc_auc'].std(),
    }
    cv_results.append(row)
    print(f"acc={row['cv_accuracy_mean']:.4f}  "
          f"f1={row['cv_f1_mean']:.4f}  "
          f"roc_auc={row['cv_roc_auc_mean']:.4f}")

df_cv = pd.DataFrame(cv_results)
print('\nCV summary:')
print(df_cv[['model','cv_accuracy_mean','cv_f1_mean','cv_roc_auc_mean']].to_string(index=False))


CV: logreg ... acc=0.6575  f1=0.6598  roc_auc=0.7238
CV: linear_svm_calibrated ... acc=0.6791  f1=0.6836  roc_auc=0.7451
CV: random_forest ... acc=0.7406  f1=0.7391  roc_auc=0.8228
CV: xgboost ... acc=0.7338  f1=0.7377  roc_auc=0.8178

CV summary:
                model  cv_accuracy_mean  cv_f1_mean  cv_roc_auc_mean
               logreg          0.657500    0.659836         0.723850
linear_svm_calibrated          0.679063    0.683567         0.745080
        random_forest          0.740625    0.739083         0.822784
              xgboost          0.733750    0.737694         0.817836


## 7) Train on full training set, evaluate on holdout

Re-fit each model on all of `X_train`, evaluate once on the holdout test set,
and save the fitted model to disk.

In [7]:
def evaluate(name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    y_proba = clf.predict_proba(X_te)[:, 1]
    return {
        'model':     name,
        'accuracy':  float(accuracy_score(y_te, y_pred)),
        'precision': float(precision_score(y_te, y_pred, zero_division=0)),
        'recall':    float(recall_score(y_te, y_pred, zero_division=0)),
        'f1':        float(f1_score(y_te, y_pred, zero_division=0)),
        'roc_auc':   float(roc_auc_score(y_te, y_proba)),
        'pr_auc':    float(average_precision_score(y_te, y_proba)),
    }, clf, y_pred, y_proba


test_results  = []
fitted_models = {}

for name, clf in models.items():
    print(f'Fitting: {name} ...', end=' ', flush=True)
    metrics, fitted_clf, y_pred, y_proba = evaluate(
        name, clf, X_train, y_train, X_test, y_test)
    test_results.append(metrics)
    fitted_models[name] = (fitted_clf, y_pred, y_proba)
    joblib.dump(fitted_clf, MODELS / f'{name}.pkl')
    print(f"acc={metrics['accuracy']:.4f}  "
          f"f1={metrics['f1']:.4f}  "
          f"roc_auc={metrics['roc_auc']:.4f}  "
          f"pr_auc={metrics['pr_auc']:.4f}")

df_test = pd.DataFrame(test_results)
print('\nHoldout results:')
print(df_test.to_string(index=False))


Fitting: logreg ... acc=0.6700  f1=0.6562  roc_auc=0.7293  pr_auc=0.7208
Fitting: linear_svm_calibrated ... acc=0.6787  f1=0.6709  roc_auc=0.7294  pr_auc=0.7485
Fitting: random_forest ... acc=0.7588  f1=0.7554  roc_auc=0.8460  pr_auc=0.8484
Fitting: xgboost ... acc=0.7688  f1=0.7673  roc_auc=0.8507  pr_auc=0.8544

Holdout results:
                model  accuracy  precision  recall       f1  roc_auc   pr_auc
               logreg   0.67000   0.684783  0.6300 0.656250 0.729281 0.720785
linear_svm_calibrated   0.67875   0.687664  0.6550 0.670935 0.729419 0.748509
        random_forest   0.75875   0.766067  0.7450 0.755387 0.846050 0.848444
              xgboost   0.76875   0.772152  0.7625 0.767296 0.850700 0.854365


## 8) Save results CSV

In [8]:
results_csv = REPORTS / 'dna_nt_test_results.csv'
df_test.to_csv(results_csv, index=False)
print('Saved:', results_csv)


Saved: /home/dpratapa/Capstone/reports/dna_nt_test_results.csv


## 9) Confusion matrices and ROC curves

In [9]:
for name, (fitted_clf, y_pred, y_proba) in fitted_models.items():

    # Confusion matrix
    cm  = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=['non-promoter','promoter']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'NT / {name}')
    plt.tight_layout()
    fig.savefig(FIGURES / f'cm_{name}.png', dpi=150)
    plt.close(fig)

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_val = roc_auc_score(y_test, y_proba)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, lw=2, label=f'AUC = {auc_val:.4f}')
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC -- NT / {name}')
    ax.legend(loc='lower right')
    plt.tight_layout()
    fig.savefig(FIGURES / f'roc_{name}.png', dpi=150)
    plt.close(fig)

print('Figures saved to:', FIGURES)


Figures saved to: /home/dpratapa/Capstone/reports/figures/dna_nt


## 10) Summary JSON

In [10]:
df_merged = df_test.merge(df_cv, on='model', how='left')
best_row   = df_test.loc[df_test['roc_auc'].idxmax()]

summary = {
    'notebook': '18_dna_nt_models',
    'embedding_source': '17_dna_nt_embeddings',
    'model': 'InstaDeepAI/nucleotide-transformer-500m-human-ref',
    'embedding_dim': int(X.shape[1]),
    'n_train': int(X_train.shape[0]),
    'n_test':  int(X_test.shape[0]),
    'cv_folds': CV_FOLDS,
    'seed': SEED,
    'best_model': {
        'name':     str(best_row['model']),
        'roc_auc':  float(best_row['roc_auc']),
        'f1':       float(best_row['f1']),
        'accuracy': float(best_row['accuracy']),
    },
    'all_results': df_merged.to_dict(orient='records'),
    'model_files': {n: str(MODELS / f'{n}.pkl') for n in models},
    'timestamp': pd.Timestamp.now().isoformat(),
}

summary_path = REPORTS / 'dna_nt_models_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print('Summary saved to:', summary_path)
print(f"Best model: {summary['best_model']['name']}  "
      f"ROC-AUC={summary['best_model']['roc_auc']:.4f}  "
      f"F1={summary['best_model']['f1']:.4f}")


Summary saved to: /home/dpratapa/Capstone/reports/dna_nt_models_summary.json
Best model: xgboost  ROC-AUC=0.8507  F1=0.7673


## 11) Full DNA paradigm comparison (all models)

Collects results from baseline, CNN, hybrid, DNABERT-2, and NT notebooks
into a single comparison table sorted by ROC-AUC.

In [11]:
result_files = {
    'baseline': REPORTS / f'dna_baseline_test_results_len{L}.csv',
    'cnn':      REPORTS / f'dna_seq_cnn_test_results_len{L}.csv',
    'hybrid':   REPORTS / f'dna_hybrid_test_results_len{L}.csv',
    'dnabert2': REPORTS / 'dna_dnabert2_test_results.csv',
    'nt':       REPORTS / 'dna_nt_test_results.csv',
}

frames = []
for paradigm, path in result_files.items():
    if path.exists():
        df_tmp = pd.read_csv(path)
        df_tmp.insert(0, 'paradigm', paradigm)
        frames.append(df_tmp)
    else:
        print(f'Not found (skipping): {path}')

if frames:
    df_all = pd.concat(frames, ignore_index=True)
    cols = [c for c in ['paradigm','model','accuracy','precision','recall','f1','roc_auc','pr_auc']
            if c in df_all.columns]
    df_all = df_all[cols].sort_values('roc_auc', ascending=False)
    print(df_all.to_string(index=False))

    comparison_csv = REPORTS / 'dna_all_paradigms_comparison.csv'
    df_all.to_csv(comparison_csv, index=False)
    print('\nUpdated comparison saved to:', comparison_csv)


Not found (skipping): /home/dpratapa/Capstone/reports/dna_hybrid_test_results_len200.csv
paradigm                 model  accuracy  precision  recall       f1  roc_auc   pr_auc
      nt               xgboost   0.76875   0.772152  0.7625 0.767296 0.850700 0.854365
dnabert2               xgboost   0.75625   0.758186  0.7525 0.755332 0.846250 0.853206
      nt         random_forest   0.75875   0.766067  0.7450 0.755387 0.846050 0.848444
dnabert2         random_forest   0.75250   0.779006  0.7050 0.740157 0.842894 0.842107
baseline         random_forest   0.74750   0.756477  0.7300 0.743003 0.837384 0.847125
     cnn               seq_cnn   0.75000   0.752525  0.7450 0.748744 0.826375 0.837063
baseline            grad_boost   0.72625   0.743935  0.6900 0.715953 0.825606 0.838261
baseline               xgboost   0.73875   0.745501  0.7250 0.735108 0.824112 0.836746
baseline                logreg   0.73000   0.740838  0.7075 0.723785 0.823444 0.833903
baseline linear_svm_calibrated   0.73375 

## Next

DNA transformer pipeline is complete. Proceed to the protein side:
- `19_protein_protbert_embeddings.ipynb` -- ProtBERT embedding extraction
- `20_protein_protbert_models.ipynb` -- classifiers on ProtBERT embeddings

After those, produce final comparison tables and figures across ALL paradigms
for both DNA and protein tasks.